# Instructor Solutions for In-Class Exercises

This notebook collects example solutions and model answers for the in-class exercises across the course notebooks.

It is intended as an instructor-only answer key. Some exercises are open-ended, so the answers below are examples of strong responses rather than the only possible correct answers.

The solutions prioritize:

- clear methodological reasoning;
- reproducible code patterns;
- responsible access decisions;
- honest documentation of limitations.


In [ ]:
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
from bs4 import BeautifulSoup


## Day 0: Parameter Discovery

Exercise prompt: create a small parameter inventory for one access route.

A good answer should show that the student understands that parameters, selectors, filters, and defaults shape the dataset before collection begins.


### Example Solution: MediaWiki API Search Parameter Inventory

| Parameter / setting | Allowed or plausible values | Default / current value | Why it matters | Documentation source |
|---|---|---|---|---|
| `action` | `query`, `help`, others | `query` | Selects the API action. For this workflow, it means we are reading existing wiki data. | MediaWiki API docs, `API:Query` |
| `list` | `search`, others | `search` | Chooses the query module. `search` returns search-result records, not full articles. | MediaWiki `API:Search` |
| `srsearch` | any search string | `digital services act` | Defines which pages can enter the search-result dataset. A different query changes the population. | MediaWiki `API:Search` |
| `srlimit` | small integer up to API limits | `5` or `50` | Batch size per request. It is not the total collection size. | MediaWiki `API:Search` |
| `sroffset` / `continue` | API-provided continuation values | provided by API | Controls pagination so the next request starts after previous results. | MediaWiki `API:Continue` |
| `format` | `json`, `xml`, etc. | `json` | Determines response format. The Python script expects JSON. | MediaWiki API docs |
| `formatversion` | `1`, `2` | `2` | Changes JSON shape. This affects parsing and reproducibility. | MediaWiki API docs |

Strong reflection: before running a collection, the student should know which parameters affect the population of records and which only affect response format.


## Day 1: API Exercise

Exercise prompt: adapt the Wikipedia API workflow by changing a query, page size, number of pages, page extracts, external links, or processed columns.


### Example Solution A: Change Query and Collection Size

Example adaptation:

- query changed from `digital services act` to `platform governance`
- page size changed from `5` to `10`
- maximum pages changed from `2` to `3`

Why it matters:

- the query changes which Wikipedia pages can enter the dataset;
- page size changes how many results are returned per request;
- max pages changes the stopping rule and therefore the maximum collection size.

What remains invisible:

- pages not surfaced by MediaWiki search ranking;
- deleted or redirected content not visible in the current response;
- content outside Wikipedia;
- page history unless explicitly requested through another endpoint/module.


In [ ]:
# Example adaptation pattern. This assumes collect_search() exists from the API notebook/script.
# It is intentionally small-scale for teaching.

query = "platform governance"
page_size = 10
max_pages = 3

# search_rows, raw_pages = collect_search(query=query, pages=max_pages, page_size=page_size)
# pd.DataFrame(search_rows).head()


### Example Solution B: Add a Processed Column

A simple useful column is a browser URL based on `pageid`.

Why meaningful:

- `pageid` is a stable identifier;
- the browser URL makes manual inspection easier;
- it links the processed table back to the visible article page.

Limitation:

- adding a URL does not make the search result a full article;
- page content still needs a second page-level API request or another access route.


In [ ]:
search_rows_example = [
    {"pageid": 123, "title": "Example page"},
    {"pageid": 456, "title": "Another example"},
]

search_df = pd.DataFrame(search_rows_example)
search_df["browser_url"] = search_df["pageid"].apply(
    lambda pageid: f"https://en.wikipedia.org/?curid={pageid}"
)
search_df


## Day 2: Static Scraping Warm-Up Exercise

Exercise prompt: adapt the quotes scraper by extracting author profile URLs, next-page URL, tags, tag count, and a provenance note.


In [ ]:
quotes_html = """
<html>
  <body>
    <div class="quote">
      <span class="text">The world as we have created it is a process of our thinking.</span>
      <span>by <small class="author">Albert Einstein</small></span>
      <a href="/author/Albert-Einstein">about</a>
      <div class="tags">
        <a class="tag" href="/tag/change/page/1/">change</a>
        <a class="tag" href="/tag/thinking/page/1/">thinking</a>
      </div>
    </div>
    <nav>
      <li class="next"><a href="/page/2/">Next</a></li>
    </nav>
  </body>
</html>
"""

base_url = "https://quotes.toscrape.com/"
quotes_soup = BeautifulSoup(quotes_html, "html.parser")

rows = []
tag_rows = []

for quote_number, quote_block in enumerate(quotes_soup.select(".quote"), start=1):
    text = quote_block.select_one(".text")
    author = quote_block.select_one(".author")
    author_link = quote_block.select_one("a[href]")
    tag_links = quote_block.select(".tags a.tag")

    tags = [tag.get_text(" ", strip=True) for tag in tag_links]

    rows.append(
        {
            "quote_number": quote_number,
            "quote": text.get_text(" ", strip=True) if text else None,
            "author": author.get_text(" ", strip=True) if author else None,
            "author_profile_url": urljoin(base_url, author_link.get("href")) if author_link else None,
            "tags": "|".join(tags),
            "tag_count": len(tags),
        }
    )

    for tag in tag_links:
        tag_rows.append(
            {
                "quote_number": quote_number,
                "tag": tag.get_text(" ", strip=True),
                "tag_url": urljoin(base_url, tag.get("href")),
            }
        )

next_link = quotes_soup.select_one("li.next a[href]")
next_page_url = urljoin(base_url, next_link.get("href")) if next_link else None

quotes_solution_df = pd.DataFrame(rows)
tags_solution_df = pd.DataFrame(tag_rows)

print("Next page:", next_page_url)
quotes_solution_df


In [ ]:
tags_solution_df


### Example Provenance Note

- Source URL: `https://quotes.toscrape.com/`
- Collection mode: static HTML scraping with `requests` and BeautifulSoup
- Main record selector: `.quote`
- Field selectors: `.text`, `.author`, `.tags a.tag`, `a[href]`
- Output files: raw HTML, quote table, tag table
- Limitation: this is a teaching site; selector stability here is not representative of many real websites.


## Day 2: MethodsNET BeautifulSoup Exercise

Exercise prompt: extract fields from a MethodsNET-style course-detail page fragment.


In [ ]:
methodsnet_detail_html = """
<article class="course-detail" data-course-code="D04">
  <h1>D04: Collecting Data from Large Online Platforms with APIs, Webscraping and DSA applications</h1>

  <section class="course-facts">
    <h3>Date &amp; Time</h3>
    <p>29.06.26 - 03.07.26</p>

    <h3>Course Time</h3>
    <p>09:00-12:00 &amp; 13:30-15:15</p>

    <h3>Instructor</h3>
    <p>Noelle Sophie Lebernegg</p>

    <h3>ECTS</h3>
    <p>4</p>
  </section>

  <section class="description">
    <h2>Short Description</h2>
    <p>This course introduces advanced techniques for collecting and managing data from VLOPs.</p>

    <h2>Learning Objectives</h2>
    <ul>
      <li>Collect data using open APIs.</li>
      <li>Build ethical and robust web scrapers.</li>
      <li>Document reproducible data collection workflows.</li>
    </ul>
  </section>
</article>
"""

methodsnet_detail_soup = BeautifulSoup(methodsnet_detail_html, "html.parser")

course_title_element = methodsnet_detail_soup.select_one("h1")
course_title = course_title_element.get_text(" ", strip=True) if course_title_element else None

course_container = methodsnet_detail_soup.select_one("article.course-detail")
course_code = course_container.get("data-course-code") if course_container else None

print("Course code:", course_code)
print("Course title:", course_title)


In [ ]:
def value_after_heading(soup, heading_text):
    heading = soup.find(
        lambda element: element.name in ["h2", "h3", "h4"]
        and element.get_text(" ", strip=True).lower() == heading_text.lower()
    )

    if heading is None:
        return None

    value_paragraph = heading.find_next("p")
    return value_paragraph.get_text(" ", strip=True) if value_paragraph else None


course_detail_row = {
    "course_code": course_code,
    "course_title": course_title,
    "date_time": value_after_heading(methodsnet_detail_soup, "Date & Time"),
    "course_time": value_after_heading(methodsnet_detail_soup, "Course Time"),
    "instructor": value_after_heading(methodsnet_detail_soup, "Instructor"),
    "ects": value_after_heading(methodsnet_detail_soup, "ECTS"),
    "short_description": value_after_heading(methodsnet_detail_soup, "Short Description"),
}

course_detail_row


In [ ]:
objective_items = methodsnet_detail_soup.select(".description li")
objectives = [item.get_text(" ", strip=True) for item in objective_items]

objective_rows = []
for number, objective in enumerate(objectives, start=1):
    objective_rows.append(
        {
            "course_code": course_code,
            "objective_number": number,
            "objective": objective,
        }
    )

objectives_df = pd.DataFrame(objective_rows)
objectives_df


### Conceptual Answers

1. The whole course record is the `article.course-detail` element.
2. The course title is in the `h1` element.
3. The course code is stored in the `data-course-code` attribute.
4. Values such as `Instructor` can be extracted by finding a heading with that text and then selecting the next paragraph.
5. Learning objectives can be stored either as a list in one course row or as a long table with one row per objective.


## Day 2: Selector Patterns Mini Exercise

### Model Answers

1. `.result.card` selects elements that have both classes: `result` and `card` on the same element.
2. `.result .card` means: select an element with class `card` inside an ancestor with class `result`. It returns nothing if the same element has both classes but there is no nested `.card` element.
3. `n_words_summary` can be created with `summary_text.split()` after extracting summary text.
4. A long article-topic table uses one row per article-topic combination.
5. Visible text comes from `.get_text(...)`; attributes come from `.get(...)`, such as `href`, `datetime`, `data-id`.
6. Machine-readable publication dates often come from a `datetime` attribute on a `<time>` element.
7. `tbody tr` is the repeated record selector because each table row in the table body corresponds to one course.
8. `cells[2]` assumes the course title is always the third cell; if the website changes column order, this breaks or extracts the wrong field.
9. A combined course label can be created as `course_code + ": " + course_title`.
10. A long learning-objectives table uses one row per objective.


In [ ]:
example_articles = pd.DataFrame(
    [
        {"article_id": "a1", "summary": "Short summary about platform governance", "topics": ["platforms", "governance"]},
        {"article_id": "a2", "summary": "Another text about online data", "topics": ["data"]},
    ]
)

example_articles["n_words_summary"] = example_articles["summary"].apply(lambda text: len(text.split()))

long_topic_rows = []
for _, row in example_articles.iterrows():
    for topic in row["topics"]:
        long_topic_rows.append({"article_id": row["article_id"], "topic": topic})

pd.DataFrame(long_topic_rows)


## Day 3: Dynamic Browser Debugging Exercise

| Scenario | Likely issue | Evidence to save | Try next | Do not do |
|---|---|---|---|---|
| `requests` returns HTML but target records are missing | Rendering issue if status is 200 and page shell is present | raw HTML, status code, content type, screenshot from manual browser | open in browser/Selenium, inspect network and DOM, wait for target selector | assume selectors are wrong before checking rendering |
| Selenium opens page but extracts zero records | Selector issue or timing issue | screenshot, rendered HTML, selector counts, diagnostics | inspect rendered HTML, wait for a more specific selector, test selector in browser dev tools | increase scale or loop endlessly |
| Scrolling does not increase count | no infinite scroll, wrong record selector, or page needs a button/cursor | scroll-count history, screenshot before/after | check whether there is a load-more button, compare page height, inspect network | keep scrolling without a stopping rule |
| Cookie/consent banner covers page | access/interface state issue | screenshot, rendered HTML, banner text | decide whether consent interaction is appropriate and document it | hide or bypass consent without reflection |
| CAPTCHA or login wall | access/compliance boundary | screenshot, status, URL, page text | stop, seek permission, use official API/research access, or choose another source | automate CAPTCHA solving, credential stuffing, identity rotation, or evasion |

Strong answer: dynamic scraping is diagnosis first, automation second.


## Day 4: DSA / Transparency Data Exercise

### Example Model Answer: `platform_name`

- Appears to measure: the platform or service that submitted or is associated with the record.
- Ambiguity: a company may operate multiple services; naming may be inconsistent across datasets; subsidiaries or product names may be used differently.
- Comparability: only comparable if the schema defines platform identity consistently and platforms report in the same way.
- What to check: official schema definition, reporting obligations, whether values are controlled vocabulary or free text, whether platform names changed over time.

### Example Model Answer: `statement_of_reasons_category`

- Appears to measure: the platform-reported reason for a moderation or transparency record.
- Ambiguity: categories may be broad, platform-controlled, or inconsistently applied; they may reflect reporting categories rather than actual decision logic.
- Comparability: limited unless all platforms use the same taxonomy and reporting rules.
- What to check: official codebook/schema, platform-specific reporting guidance, missingness, category changes over time.

Strong answer: transparency data should be treated as a structured reporting artifact, not direct access to full platform behavior.


## Day 4: Data Quality Audit Exercise


In [ ]:
reference = pd.DataFrame(
    [
        {"post_id": "p1", "text": "A", "timestamp": "2026-01-01", "author_id": "u1"},
        {"post_id": "p2", "text": "B", "timestamp": "2026-01-02", "author_id": "u2"},
        {"post_id": "p3", "text": "C", "timestamp": "2026-01-03", "author_id": "u3"},
    ]
)

observed = pd.DataFrame(
    [
        {"post_id": "p1", "text": "A", "timestamp": None},
        {"post_id": "p3", "text": "C", "timestamp": None},
        {"post_id": "p4", "text": "D", "timestamp": None},
    ]
)

ref_ids = set(reference["post_id"])
obs_ids = set(observed["post_id"])

missing_from_observed = ref_ids - obs_ids
extra_in_observed = obs_ids - ref_ids
common_ids = ref_ids & obs_ids

print("Missing from observed:", missing_from_observed)
print("Only in observed:", extra_in_observed)
print("Common IDs:", common_ids)
print("Observed missingness:")
print(observed.isna().sum())


### Model Methods-Section Language

The observed dataset did not fully reproduce the reference set. Two of three reference IDs were present, one reference ID was missing, and one additional observed ID appeared only in the observed source. Timestamp metadata were missing for all observed records. Therefore, analyses requiring complete temporal ordering or full population coverage would not be appropriate. Analyses limited to content fields among overlapping IDs may still be possible, but results should be described as conditional on the platform access route and its missing metadata.


## Day 5: AI-Assisted Collection Exercise

### Example Solution: Selector Suggestions from a Public HTML Excerpt

Task asked of AI:

> Given this small public HTML excerpt, suggest CSS selectors for article title, URL, date, and summary. Do not infer anything beyond the excerpt.

Evidence provided:

- a small non-sensitive HTML snippet;
- no credentials;
- no cookies;
- no personal/private data;
- no full paywalled article text.

Verification:

- checked suggested selectors manually in BeautifulSoup;
- tested selectors on three saved pages;
- counted missing fields;
- rejected any selector based only on element position, such as `div:nth-child(3)`, unless no better option exists and the fragility is documented.

Provenance note:

> AI assistance was used to propose candidate selectors from a public HTML excerpt. All selectors were manually verified against saved HTML before use. No credentials, cookies, or private/personal data were shared. AI-generated suggestions were treated as draft assistance and not as authoritative documentation.


## Day 5: Practical Collection Workflows Exercise

### Example Solution: MethodsNET Course Scraper

1. Reasonable run frequency: manually before the course or at most weekly during course-planning periods. Daily/hourly scheduling would be unnecessary.
2. Irresponsible scheduling: frequent repeated scraping without checking robots.txt, terms, rate limits, or whether the site changes rarely.
3. Outputs:
   - `raw/`: saved course-list HTML and detail-page HTML
   - `processed/`: course links CSV, course details CSV
   - `logs/`: run log with timestamps, warnings, errors
   - `reports/`: provenance, diagnostics, manifest
4. Monitoring checks:
   - number of course links found is not zero;
   - expected columns exist;
   - detail pages parsed is close to requested number;
   - error count is below threshold;
   - raw HTML was saved.
5. Manifest fields:
   - run label, timestamp, script path, git commit if available, command, parameters, input URLs, output files, record counts, warnings/errors, software versions.
6. Git/archive policy:
   - commit scripts, configs, small synthetic examples, and documentation;
   - do not commit large or frequently changing raw outputs unless they are small and appropriate;
   - archive larger run outputs separately with clear version labels.


## Day 5: Reproducible Project Exercise


In [ ]:
config = {
    "collector": "wikipedia_search",
    "query": "platform governance",
    "page_size": 10,
    "pages": 2,
    "output_dir": "data",
    "limitations": [
        "Search results depend on MediaWiki ranking and current page state.",
        "This is not a complete corpus of all platform-governance content.",
    ],
    "monitoring_checks": [
        "processed row count greater than zero",
        "raw response file exists",
        "required columns present: pageid, title, timestamp",
    ],
}

config


### Example Answers

New processed field:

- `browser_url = "https://en.wikipedia.org/?curid=" + pageid`

Limitation to add to manifest:

- Search-result ranking is platform/API controlled and may change over time.

What someone needs to reproduce the run:

- repository version / commit;
- script path;
- Python environment and package versions;
- exact query, page size, page count, endpoint, and date/time;
- raw responses;
- processed output paths;
- provenance/manifest file.

Monitoring check:

- fail or warn if processed CSV has zero rows;
- warn if expected columns are missing;
- warn if status-code errors occurred;
- warn if duplicate IDs exceed expected level.

Cron expression for weekly run:

```text
0 9 * * 1 cd /path/to/repo && python scripts/runnable_workflows/01_api_wikipedia.py --query "platform governance" --pages 2 --page-size 10 --outdir data >> data/logs/wiki_collect.log 2>&1
```

Why scheduling without monitoring is risky:

A scheduled scraper can silently collect empty pages, blocked pages, login pages, duplicated records, or changed HTML for weeks. Automation makes errors faster and less visible unless logs and checks are part of the workflow.
